# Public Transport Delay Prediction using Machine Learning

This notebook transforms the raw public transport dataset into clean, leakage-aware feature sets for:

- **Classification** — predict whether a trip will be delayed.
- **Regression** — predict the number of minutes of arrival delay.
- **Time-series analysis/forecasting** — preserve chronological information for temporal modelling.
- **Feature engineering practice** — combine scheduling, weather, traffic and event information.

The preprocessing strategy is designed so that information available only after a journey does not become a predictor of the pre-trip classification task.

## Objectives

The main objectives are to:

1. Reload the raw dataset independently of Notebook 1.
2. Identify target and leakage variables for each modelling task.
3. Convert date/time fields into useful numerical and categorical features.
4. Engineer event-related and scheduling features.
5. Handle the missing `event_type` values appropriately.
6. Create a common modelling feature table.
7. Separate classification and regression targets.
8. Split supervised data before fitting learned preprocessing steps.
9. Build reusable preprocessing pipelines for numerical and categorical features.
10. Prepare a chronological dataset for time-series modelling.

In [46]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.impute import SimpleImputer

%matplotlib inline

In [47]:
# Load the raw dataset 

df = pd.read_csv("../data/public_transport_delays.csv")

print("Dataset shape:", df.shape)
df.head()

Dataset shape: (2000, 24)


,trip_id,date,time,transport_type,route_id,origin_station,destination_station,scheduled_departure,scheduled_arrival,actual_departure_delay_min,...,wind_speed_kmh,precipitation_mm,event_type,event_attendance_est,traffic_congestion_index,holiday,peak_hour,weekday,season,delayed
0,T00000,2023-01-01,05:00:00,Tram,Route_15,Station_31,Station_6,05:02:00,05:55:00,12,...,46,13.0,NaN,500,81,0,1,6,Winter,0
1,T00001,2023-01-01,05:15:00,Metro,Route_12,Station_49,Station_32,05:16:00,05:55:00,15,...,11,11.4,NaN,0,53,0,0,6,Autumn,1
2,T00002,2023-01-01,05:30:00,Bus,Route_16,Station_29,Station_42,05:33:00,06:17:00,0,...,31,14.1,Sports,0,67,1,0,6,Autumn,0
3,T00003,2023-01-01,05:45:00,Tram,Route_19,Station_26,Station_18,05:49:00,06:08:00,15,...,41,6.4,NaN,500,84,0,0,6,Winter,1
4,T00004,2023-01-01,06:00:00,Tram,Route_8,Station_18,Station_15,06:00:00,06:35:00,-1,...,30,18.5,NaN,500,46,0,0,6,Spring,1


## Prediction Scenarios

The dataset supports different modelling tasks, so the target and permissible predictors must be defined separately.

### Classification

Predict:

`delayed` → 0 = on time, 1 = delayed

For a realistic pre-trip prediction, post-outcome delay measurements must not be used as predictors.

### Regression

Predict:

`actual_arrival_delay_min`

This is a regression target because the objective is to estimate the number of minutes of arrival delay. The target itself must not be included among the input features.

### Time Series

The chronological structure of the observations will be preserved separately. Time-series analysis will use dates and aggregated delay measures rather than randomly shuffled observations.

In [48]:
classification_target = "delayed"
regression_target = "actual_arrival_delay_min"

leakage_columns = [
    "actual_departure_delay_min",
    "actual_arrival_delay_min"
]

identifier_columns = [
    "trip_id"
]

print("Classification target:", classification_target)
print("Regression target:", regression_target)
print("Post-outcome variables excluded from predictive features:", leakage_columns)

Classification target: delayed
Regression target: actual_arrival_delay_min
Post-outcome variables excluded from predictive features: ['actual_departure_delay_min', 'actual_arrival_delay_min']


## Datetime Feature Engineering

The raw dataset stores date and time information as strings. Rather than passing raw timestamps directly to the models, useful calendar and schedule features are extracted.

The scheduled departure and arrival times are used to derive:

- departure hour
- departure minute
- scheduled trip duration
- day of week
- weekend indicator
- month
- day of month
- quarter

Cyclical encoding will also be created for departure time because hours close to midnight and midnight are temporally close.

In [49]:
# Parse date and time columns

df["date"] = pd.to_datetime(df["date"])

df["time"] = pd.to_datetime(
    df["time"],
    format="%H:%M:%S"
)

df["scheduled_departure"] = pd.to_datetime(
    df["scheduled_departure"],
    format="%H:%M:%S"
)

df["scheduled_arrival"] = pd.to_datetime(
    df["scheduled_arrival"],
    format="%H:%M:%S"
)

# Calendar features
df["year"] = df["date"].dt.year
df["month"] = df["date"].dt.month
df["day_of_month"] = df["date"].dt.day
df["day_of_week"] = df["date"].dt.day_name()
df["weekday_num"] = df["date"].dt.dayofweek
df["is_weekend"] = (df["weekday_num"] >= 5).astype(int)
df["quarter"] = df["date"].dt.quarter

# Scheduled time features
df["departure_hour"] = df["scheduled_departure"].dt.hour
df["departure_minute"] = df["scheduled_departure"].dt.minute
df["arrival_hour"] = df["scheduled_arrival"].dt.hour
df["arrival_minute"] = df["scheduled_arrival"].dt.minute

# Minutes since midnight
df["departure_minutes"] = (
    df["departure_hour"] * 60 +
    df["departure_minute"]
)

df["arrival_minutes"] = (
    df["arrival_hour"] * 60 +
    df["arrival_minute"]
)

# Scheduled trip duration, including trips crossing midnight
df["scheduled_trip_duration_min"] = (
    df["arrival_minutes"] - df["departure_minutes"]
) % (24 * 60)

# Cyclical representation of departure time
df["departure_hour_sin"] = np.sin(
    2 * np.pi * df["departure_minutes"] / (24 * 60)
)

df["departure_hour_cos"] = np.cos(
    2 * np.pi * df["departure_minutes"] / (24 * 60)
)

print("Datetime feature engineering completed.")
df[
    [
        "date",
        "day_of_week",
        "is_weekend",
        "departure_hour",
        "departure_minute",
        "scheduled_trip_duration_min",
        "departure_hour_sin",
        "departure_hour_cos"
    ]
].head()

Datetime feature engineering completed.


,date,day_of_week,is_weekend,departure_hour,departure_minute,scheduled_trip_duration_min,departure_hour_sin,departure_hour_cos
0,2023-01-01,Sunday,1,5,2,53,0.968148,2.503800e-01
1,2023-01-01,Sunday,1,5,16,39,0.981627,1.908090e-01
2,2023-01-01,Sunday,1,5,33,44,0.993068,1.175374e-01
3,2023-01-01,Sunday,1,5,49,19,0.998848,4.797813e-02
4,2023-01-01,Sunday,1,6,0,35,1.000000,6.123234e-17


## Event Feature Engineering

EDA showed that `event_type` contains substantial missingness and that some missing event types still have positive event attendance. Therefore, missing `event_type` values cannot safely be interpreted as "No Event".

They are represented as:

`Unknown / Missing Event`

A separate binary feature, `has_event`, will be derived from event attendance.

Because event attendance is highly right-skewed, a logarithmic transformation will also be created to reduce the influence of very large events.

In [50]:
df["event_type"] = df["event_type"].fillna("Unknown / Missing Event")

df["has_event"] = (
    df["event_attendance_est"] > 0
).astype(int)

df["log_event_attendance"] = np.log1p(
    df["event_attendance_est"]
)

print("Event feature engineering completed.")
df[
    [
        "event_type",
        "event_attendance_est",
        "has_event",
        "log_event_attendance"
    ]
].head()

Event feature engineering completed.


,event_type,event_attendance_est,has_event,log_event_attendance
0,Unknown / Missing Event,500,1,6.216606
1,Unknown / Missing Event,0,0,0.000000
2,Sports,0,0,0.000000
3,Unknown / Missing Event,500,1,6.216606
4,Unknown / Missing Event,500,1,6.216606


## Scheduling and Context Features

Several existing variables already represent useful operational context.

`peak_hour` is retained because it directly represents whether the scheduled trip occurs during a peak period.

`holiday` is retained as a binary indicator.

`weekday` is available in the raw dataset, but the calendar-derived `day_of_week` and `weekday_num` provide a clearer representation of the same weekly information. To avoid redundant representations, the original `weekday` field will not be used in the supervised feature matrix.

In [51]:
# Check consistency between the raw weekday field and the date-derived weekday

weekday_check = (
    df["weekday"] == df["weekday_num"]
).mean()

print(f"Agreement between raw weekday and date-derived weekday: {weekday_check:.2%}")

# Inspect the engineered feature set
df[
    [
        "date",
        "weekday",
        "weekday_num",
        "day_of_week",
        "is_weekend",
        "peak_hour",
        "holiday"
    ]
].head()

Agreement between raw weekday and date-derived weekday: 100.00%


,date,weekday,weekday_num,day_of_week,is_weekend,peak_hour,holiday
0,2023-01-01,6,6,Sunday,1,1,0
1,2023-01-01,6,6,Sunday,1,0,0
2,2023-01-01,6,6,Sunday,1,0,1
3,2023-01-01,6,6,Sunday,1,0,0
4,2023-01-01,6,6,Sunday,1,0,0


## Review the Engineered Dataset

At this stage, the dataframe contains the original variables plus engineered features.

The next step is to define the modelling feature groups explicitly rather than allowing the model to consume every available column. This is important because the dataset contains post-outcome delay measurements that would cause target leakage.

In [52]:
print("Rows:", df.shape[0])
print("Columns:", df.shape[1])

print("Engineered columns:")
engineered_columns = [
    "year",
    "month",
    "day_of_month",
    "day_of_week",
    "weekday_num",
    "is_weekend",
    "quarter",
    "departure_hour",
    "departure_minute",
    "arrival_hour",
    "arrival_minute",
    "departure_minutes",
    "arrival_minutes",
    "scheduled_trip_duration_min",
    "departure_hour_sin",
    "departure_hour_cos",
    "has_event",
    "log_event_attendance"
]

print(engineered_columns)

Rows: 2000
Columns: 42
Engineered columns:
['year', 'month', 'day_of_month', 'day_of_week', 'weekday_num', 'is_weekend', 'quarter', 'departure_hour', 'departure_minute', 'arrival_hour', 'arrival_minute', 'departure_minutes', 'arrival_minutes', 'scheduled_trip_duration_min', 'departure_hour_sin', 'departure_hour_cos', 'has_event', 'log_event_attendance']


## Define Common Predictive Features

The common feature set contains information that can reasonably be available before or around the scheduled trip.

### Categorical features

- transport type
- route
- origin station
- destination station
- weather condition
- event type
- season
- day of week

### Numerical features

- weather measurements
- event attendance transformation
- traffic congestion
- holiday and peak-hour indicators
- engineered scheduling features

### Explicitly excluded

- `trip_id` — identifier only
- `actual_departure_delay_min` — post-outcome information
- `actual_arrival_delay_min` — post-outcome information / regression target
- raw `date`, `time` and schedule timestamps — represented through engineered features
- raw `weekday` — redundant with date-derived weekday

In [53]:
categorical_features = [
    "transport_type",
    "route_id",
    "origin_station",
    "destination_station",
    "weather_condition",
    "event_type",
    "season",
    "day_of_week"
]

numerical_features = [
    "temperature_C",
    "humidity_percent",
    "wind_speed_kmh",
    "precipitation_mm",
    "traffic_congestion_index",
    "holiday",
    "peak_hour",
    "event_attendance_est",
    "log_event_attendance",
    "has_event",
    "year",
    "month",
    "day_of_month",
    "weekday_num",
    "is_weekend",
    "quarter",
    "departure_hour",
    "departure_minute",
    "arrival_hour",
    "arrival_minute",
    "departure_minutes",
    "arrival_minutes",
    "scheduled_trip_duration_min",
    "departure_hour_sin",
    "departure_hour_cos"
]

feature_columns = categorical_features + numerical_features

print("Number of categorical features:", len(categorical_features))
print("Number of numerical features:", len(numerical_features))
print("Total predictive features:", len(feature_columns))

Number of categorical features: 8
Number of numerical features: 25
Total predictive features: 33


## Check Missing Values in the Modelling Features

The main missing categorical variable identified during EDA, `event_type`, has already been converted to `Unknown / Missing Event`.

This check confirms the state of the final feature table before splitting the supervised datasets.

In [54]:
missing_features = (
    df[feature_columns]
    .isna()
    .sum()
    .sort_values(ascending=False)
)

missing_features[missing_features > 0]

Series([], dtype: int64)

## Classification Dataset

For classification, the target is `delayed`.

The post-outcome delay columns will be excluded from `X_classification`.

A stratified train-test split will be used so that the proportion of delayed and non-delayed trips remains similar across the two sets.

In [55]:
X_classification = df[feature_columns].copy()
y_classification = df[classification_target].copy()

X_train_cls, X_test_cls, y_train_cls, y_test_cls = train_test_split(
    X_classification,
    y_classification,
    test_size=0.20,
    random_state=42,
    stratify=y_classification
)

print("Classification training shape:", X_train_cls.shape)
print("Classification test shape:", X_test_cls.shape)

print("Training target distribution:")
print(y_train_cls.value_counts(normalize=True).sort_index().round(3))

print("Test target distribution:")
print(y_test_cls.value_counts(normalize=True).sort_index().round(3))


Classification training shape: (1600, 33)
Classification test shape: (400, 33)
Training target distribution:
delayed
0    0.251
1    0.749
Name: proportion, dtype: float64
Test target distribution:
delayed
0    0.25
1    0.75
Name: proportion, dtype: float64


## Regression Dataset

For regression, the target is `actual_arrival_delay_min`.

The target will be separated from the predictors, and both actual delay columns will be excluded from the input feature set. This prevents the regression model from using the departure delay as a shortcut to predict arrival delay.

The same common pre-trip features will be therefore used as predictors for the regression task.

In [56]:
X_regression = df[feature_columns].copy()
y_regression = df[regression_target].copy()

X_train_reg, X_test_reg, y_train_reg, y_test_reg = train_test_split(
    X_regression,
    y_regression,
    test_size=0.20,
    random_state=42
)

print("Regression training shape:", X_train_reg.shape)
print("Regression test shape:", X_test_reg.shape)

print("Regression target summary:")
print(y_train_reg.describe())

Regression training shape: (1600, 33)
Regression test shape: (400, 33)
Regression target summary:
count    1600.000000
mean       13.393750
std         9.314864
min        -3.000000
25%         6.000000
50%        13.000000
75%        22.000000
max        29.000000
Name: actual_arrival_delay_min, dtype: float64


## Classification Preprocessing Pipeline

Preprocessing will be fitted only on the training data.

### Numerical variables

Missing values will be median-imputed and numerical variables will be standardised.

### Categorical variables

Missing values will be filled using the most frequent category and categorical variables will be one-hot encoded.

`handle_unknown="ignore"` ensures that a category appearing in the test set but not in training does not cause the transformation to fail.

This pipeline prevents preprocessing information from the test set being used during model training.

In [57]:
numeric_transformer = Pipeline(
    steps=[
        ("imputer", SimpleImputer(strategy="median")),
        ("scaler", StandardScaler())
    ]
)

categorical_transformer = Pipeline(
    steps=[
        ("imputer", SimpleImputer(strategy="most_frequent")),
        ("onehot", OneHotEncoder(
            handle_unknown="ignore",
            sparse_output=False
        ))
    ]
)

preprocessor = ColumnTransformer(
    transformers=[
        ("num", numeric_transformer, numerical_features),
        ("cat", categorical_transformer, categorical_features)
    ],
    remainder="drop"
)

X_train_cls_processed = preprocessor.fit_transform(X_train_cls)
X_test_cls_processed = preprocessor.transform(X_test_cls)

print("Processed training shape:", X_train_cls_processed.shape)
print("Processed test shape:", X_test_cls_processed.shape)

Processed training shape: (1600, 172)
Processed test shape: (400, 172)


### Missing-Value Handling

The raw dataset contains missing values only in `event_type`. These were investigated
during EDA and converted to `Unknown / Missing Event` because missing event types
cannot reliably be interpreted as the absence of an event.

The remaining modelling features contain no missing values. Median and
most-frequent imputers are retained in the preprocessing pipelines as a
defensive measure so that the pipeline remains robust to missing values in
future/unseen data.

## Inspect the Processed Feature Matrix

One-hot encoding expands categorical variables into binary indicator columns.

The transformed feature names will be extracted so that the final feature matrix remains interpretable when feature importance or SHAP analysis is performed later.

In [58]:
feature_names = preprocessor.get_feature_names_out()

print("Number of processed features:", len(feature_names))

clss_preview = pd.DataFrame(
    X_train_cls_processed[:5],
    columns=feature_names
)

clss_preview.head()


Number of processed features: 172


,num__temperature_C,num__humidity_percent,num__wind_speed_kmh,num__precipitation_mm,num__traffic_congestion_index,num__holiday,num__peak_hour,num__event_attendance_est,num__log_event_attendance,num__has_event,...,cat__season_Spring,cat__season_Summer,cat__season_Winter,cat__day_of_week_Friday,cat__day_of_week_Monday,cat__day_of_week_Saturday,cat__day_of_week_Sunday,cat__day_of_week_Thursday,cat__day_of_week_Tuesday,cat__day_of_week_Wednesday
0,1.002868,-1.213025,-0.443770,-0.436638,0.189668,-0.31208,-0.604308,-0.426391,-0.7751,-0.800641,...,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0
1,-1.265714,0.406755,-0.154528,1.098034,0.224145,3.20431,-0.604308,-0.426391,-0.7751,-0.800641,...,0.0,0.0,1.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0
2,1.167384,-0.280424,1.465230,-0.281446,0.534440,-0.31208,-0.604308,-0.426391,-0.7751,-0.800641,...,1.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0
3,-0.018860,-0.378593,-1.716437,1.684313,0.499963,-0.31208,1.654786,-0.426391,-0.7751,-0.800641,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0
4,-1.672674,0.750345,-1.485043,-0.557342,1.430849,-0.31208,-0.604308,-0.393813,0.6671,1.249000,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0


## Regression Preprocessing Pipeline

The regression task uses the same preprocessing logic. A separate fitted transformer will be used so that the regression workflow remains independent and reproducible.

In [59]:
regression_preprocessor = ColumnTransformer(
    transformers=[
        ("num", numeric_transformer, numerical_features),
        ("cat", categorical_transformer, categorical_features)
    ],
    remainder="drop"
)

X_train_reg_processed = regression_preprocessor.fit_transform(X_train_reg)
X_test_reg_processed = regression_preprocessor.transform(X_test_reg)

print("Processed regression training shape:", X_train_reg_processed.shape)
print("Processed regression test shape:", X_test_reg_processed.shape)

Processed regression training shape: (1600, 172)
Processed regression test shape: (400, 172)


In [61]:
# Get processed feature names for regression

reg_feature_names = regression_preprocessor.get_feature_names_out()

print(
    "Number of processed regression features:",
    len(reg_feature_names)
)

reg_preview = pd.DataFrame(
    X_train_reg_processed[:5],
    columns=reg_feature_names
)

reg_preview.head()

Number of processed regression features: 172


,num__temperature_C,num__humidity_percent,num__wind_speed_kmh,num__precipitation_mm,num__traffic_congestion_index,num__holiday,num__peak_hour,num__event_attendance_est,num__log_event_attendance,num__has_event,...,cat__season_Spring,cat__season_Summer,cat__season_Winter,cat__day_of_week_Friday,cat__day_of_week_Monday,cat__day_of_week_Saturday,cat__day_of_week_Sunday,cat__day_of_week_Thursday,cat__day_of_week_Tuesday,cat__day_of_week_Wednesday
0,-1.240122,-1.603871,-0.269503,1.400654,-1.379675,-0.315684,-0.617813,-0.433186,-0.784629,-0.811191,...,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0
1,0.614470,0.016170,-1.544745,0.815767,1.316702,-0.315684,-0.617813,-0.433186,-0.784629,-0.811191,...,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0
2,0.718954,-0.671120,1.121669,1.022198,0.872995,-0.315684,-0.617813,-0.433186,-0.784629,-0.811191,...,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0
3,-0.717702,-0.720213,-1.544745,1.452262,-0.697048,-0.315684,-0.617813,-0.433186,-0.784629,-0.811191,...,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0
4,1.223960,0.801644,0.715910,-1.231338,0.361024,-0.315684,-0.617813,-0.433186,-0.784629,-0.811191,...,0.0,1.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0


## Class Imbalance — Classification

The classification target is imbalanced, with delayed trips forming the majority class.

Rather than applying SMOTE before the train-test split, imbalance handling will be performed later within the modelling workflow.

For models that support it, `class_weight="balanced"` can be used. If SMOTE is evaluated, it must be applied **only to the training data** after the train-test split.

We are therefore preparing clean training and test sets without contaminating the test set.

In [62]:
class_distribution = (
    y_classification
    .value_counts()
    .rename(index={0: "On Time", 1: "Delayed"})
)

class_distribution

delayed
Delayed    1499
On Time     501
Name: count, dtype: int64

## Time-Series Dataset Preparation

Time-series modelling requires chronological ordering rather than a random train-test split.

The original date will be therefore preserved in a separate time-series dataframe.

Daily aggregates will be created to support analysis and forecasting of:

- number of trips
- number of delayed trips
- delay rate
- average arrival delay

The time-series dataset will be sorted chronologically and should be split by time in the dedicated forecasting notebook.

In [63]:
ts_df = df.copy()

daily_ts = (
    ts_df
    .groupby("date")
    .agg(
        trips=("trip_id", "count"),
        delayed_trips=("delayed", "sum"),
        delay_rate=("delayed", "mean"),
        average_arrival_delay_min=("actual_arrival_delay_min", "mean")
    )
    .reset_index()
    .sort_values("date")
)

daily_ts["delay_rate"] = daily_ts["delay_rate"] * 100

daily_ts.head()

,date,trips,delayed_trips,delay_rate,average_arrival_delay_min
0,2023-01-01,76,54,71.052632,12.013158
1,2023-01-02,96,83,86.458333,15.000000
2,2023-01-03,96,73,76.041667,13.812500
3,2023-01-04,96,76,79.166667,13.916667
4,2023-01-05,96,64,66.666667,12.031250


In [64]:
print("Time-series observations:", len(daily_ts))
print("Start date:", daily_ts["date"].min().date())
print("End date:", daily_ts["date"].max().date())

daily_ts.tail()

Time-series observations: 22
Start date: 2023-01-01
End date: 2023-01-22


,date,trips,delayed_trips,delay_rate,average_arrival_delay_min
17,2023-01-18,96,69,71.875000,12.364583
18,2023-01-19,96,74,77.083333,13.510417
19,2023-01-20,96,72,75.000000,14.697917
20,2023-01-21,96,79,82.291667,14.333333
21,2023-01-22,4,4,100.000000,19.500000


## Chronological Time-Series Split

Because the dataset contains only 22 days, the forecasting dataset is small. Therefore, in the time-series section we will treat forecasting as an applied demonstration rather than making broad long-term claims.

The final 20% of chronological observations will be reserved as a test period. No future observations will be used to construct the training period.

In [65]:
split_index = int(len(daily_ts) * 0.80)

ts_train = daily_ts.iloc[:split_index].copy()
ts_test = daily_ts.iloc[split_index:].copy()

print("Time-series training observations:", len(ts_train))
print("Time-series test observations:", len(ts_test))

print("Training period:")
print(ts_train["date"].min().date(), "to", ts_train["date"].max().date())

print("Test period:")
print(ts_test["date"].min().date(), "to", ts_test["date"].max().date())

Time-series training observations: 17
Time-series test observations: 5
Training period:
2023-01-01 to 2023-01-17
Test period:
2023-01-18 to 2023-01-22


## Final Feature Summary

The feature-engineering stage has produced a common predictive feature set for supervised learning and a separate chronological dataset for time-series modelling.

### Classification

**Target:** `delayed`

Uses the leakage-free common feature set.

### Regression

**Target:** `actual_arrival_delay_min`

Uses the same contextual features, with the delay target separated from the predictors.

### Time Series

Uses chronological daily aggregates and does not use random shuffling.

### Key engineered features

- `departure_hour`
- `departure_minute`
- `scheduled_trip_duration_min`
- `day_of_week`
- `weekday_num`
- `is_weekend`
- `departure_hour_sin`
- `departure_hour_cos`
- `has_event`
- `log_event_attendance`

### Leakage controls

The following post-outcome variables are excluded from supervised predictors:

- `actual_departure_delay_min`
- `actual_arrival_delay_min`

The regression target is allowed to be `actual_arrival_delay_min`, but it is never included as an input feature.

In [67]:
print("=" * 60)
print("FINAL PREPROCESSING SUMMARY")
print("=" * 60)

print(f"Raw dataset: {df.shape[0]:,} rows × 24 columns")
print(f"Supervised predictive features: {len(feature_columns)}")
print(f"Categorical features: {len(categorical_features)}")
print(f"Numerical features: {len(numerical_features)}")
print(f"Processed classification features: {X_train_cls_processed.shape[1]}")
print(f"Processed regression features: {X_train_reg_processed.shape[1]}")
print(f"Time-series daily observations: {len(daily_ts)}")

print("Excluded post-outcome variables:")
for col in leakage_columns:
    print(f"- {col}")


FINAL PREPROCESSING SUMMARY
Raw dataset: 2,000 rows × 24 columns
Supervised predictive features: 33
Categorical features: 8
Numerical features: 25
Processed classification features: 172
Processed regression features: 172
Time-series daily observations: 22
Excluded post-outcome variables:
- actual_departure_delay_min
- actual_arrival_delay_min
